# Featurization: SMILES to Dense Graph tensors and back

QM9 pipeline: raw SMILES to dense graph
tensors (X, E) and then reconstruct SMILES

## Setup (Colab)
Clone the repo and let `uv` install the package + all dependencies declared in
`pyproject.toml` (rdkit, torch_molecule, …). Run once per Colab session.

In [ ]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

# uv installs the package (editable) + every dependency listed in pyproject.toml
!pip install -q uv
!uv pip install --system -q -e .
print("cwd:", os.getcwd())

## The encoding
A molecule with `N` heavy atoms becomes two dense one-hot tensors:

- **`X` — `(N, 4)`** atom types over the QM9 vocab `{C, N, O, F}`.
- **`E` — `(N, N, 4)`** bond types over `{no-bond, single, double, triple}`, symmetric,
  with `no-bond` (class 0) on the diagonal.

Molecules are **Kekulized** first, so aromatic rings become alternating single/double bonds
— there is no separate aromatic class (the DiGress / CatFlow convention).

In [ ]:
from data import smiles_to_tensor, tensor_to_mol, QM9_ATOMS
from rdkit import Chem
from rdkit.Chem import Draw

smi = "CC(=O)O"  # acetic acid
X, E = smiles_to_tensor(smi)

print(f"X {X.shape}  E {E.shape}")
print("atoms:", [QM9_ATOMS[i] for i in X.argmax(-1)])
print("bond-class adjacency (0=none, 1=single, 2=double, 3=triple):")
print(E.argmax(-1))

## Round-trip: tensors → molecule
`tensor_to_mol` inverts the encoding and sanitizes; we compare RDKit canonical SMILES on
both sides. Drawing the original next to the reconstruction is the visual proof.

In [ ]:
def canon(s):
    # non-isomeric: our featurizer doesn't encode stereochemistry, so compare flat graphs
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None

rebuilt = tensor_to_mol(X, E)
print("original    :", canon(smi))
print("reconstructed:", Chem.MolToSmiles(rebuilt, isomericSmiles=False))
print("match:", Chem.MolToSmiles(rebuilt, isomericSmiles=False) == canon(smi))

Draw.MolsToGridImage(
    [Chem.MolFromSmiles(smi), rebuilt],
    legends=["original", "reconstructed"],
    molsPerRow=2, subImgSize=(260, 220),
)

In [ ]:
# Curated round-trip checks (manual confirmation) — every row should print match=True.
# Covers single/double/triple bonds, fluorines, and Kekulized aromatic rings.
checks = ["CCO", "CC(=O)O", "C#N", "CC#N", "FC(F)(F)F", "c1ccncc1", "Oc1ccccc1", "N#Cc1ccccc1"]
for s in checks:
    X_c, E_c = smiles_to_tensor(s)
    m = tensor_to_mol(X_c, E_c)
    got = Chem.MolToSmiles(m) if m is not None else None
    print(f"{s:14s} -> {str(got):14s}  match={m is not None and got == canon(s)}")

## End-to-end over real QM9
Checkbox-1 evidence: featurize → reconstruct a sample of real QM9, report the round-trip rate.
Charged species (amino-acid zwitterions, ~0.8%) fall outside the neutral {C,N,O,F} vocab and
are counted as **dropped** (excluded), separately from genuine round-trip failures.

In [ ]:
from torch_molecule.datasets import load_qm9

qm9 = load_qm9(local_dir="./data")
sample = list(qm9.data[:5000])

ok, dropped, fails = 0, [], []
for s in sample:
    try:
        X_i, E_i = smiles_to_tensor(s)
    except Exception:
        dropped.append(s)          # charged / out-of-vocab atom (amino-acid zwitterions) -> excluded
        continue
    m = tensor_to_mol(X_i, E_i)
    if m is not None and Chem.MolToSmiles(m) == canon(s):
        ok += 1
    else:
        fails.append(s)

kept = len(sample) - len(dropped)
print(f"dropped (charged / out-of-vocab): {len(dropped)}/{len(sample)} = {len(dropped) / len(sample):.2%}")
print(f"round-trip on kept:               {ok}/{kept} = {ok / kept:.2%}" if kept else "nothing kept")
print(f"{len(fails)} genuine round-trip failures; first few: {fails[:10]}")

## ZINC-250k
the 9 neutral heavy-atom types `{C,N,O,F,P,S,Cl,Br,I}` plus **`N+`** and **`O-`** as standalone atom
types. Molecules with any other formal charge are dropped, and stereochemistry is ignored (we
compare non-isomeric SMILES).

In [ ]:
# Curated ZINC checks: exercise the wider vocab + charged atom types (N+, O-).
from data import ZINC_ATOMS

zinc_checks = ["CC[N+](C)(C)C", "CC(=O)[O-]", "Clc1ccccc1", "c1ccsc1", "Brc1ccccc1", "FC(F)F"]
for s in zinc_checks:
    X_z, E_z = smiles_to_tensor(s, atom_vocab=ZINC_ATOMS)
    m = tensor_to_mol(X_z, E_z, atom_vocab=ZINC_ATOMS)
    got = Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None
    print(f"{s:18s} -> {str(got):18s}  match={m is not None and got == canon(s)}")

In [ ]:
from torch_molecule.datasets import load_zinc250k

zinc = load_zinc250k(local_dir="./data")
zsample = list(zinc.data[:5000])

ok, dropped, fails = 0, [], []
for s in zsample:
    try:
        X_i, E_i = smiles_to_tensor(s, atom_vocab=ZINC_ATOMS)
    except Exception:
        dropped.append(s)          # other-charge / out-of-vocab atom -> excluded 
        continue
    m = tensor_to_mol(X_i, E_i, atom_vocab=ZINC_ATOMS)
    if m is not None and Chem.MolToSmiles(m, isomericSmiles=False) == canon(s):
        ok += 1
    else:
        fails.append(s)

kept = len(zsample) - len(dropped)
print(f"dropped (other-charge / out-of-vocab): {len(dropped)}/{len(zsample)} = {len(dropped) / len(zsample):.2%}")
print(f"round-trip on kept:                    {ok}/{kept} = {ok / kept:.2%}" if kept else "nothing kept")
print(f"{len(fails)} round-trip failures; first few: {fails[:10]}")

## Coverage over the full datasets
How much of each *full* dataset the vocab can represent

In [ ]:
# Full-dataset drop counts (reuses qm9 / zinc loaded above)
def count_dropped(smiles_list, vocab):
    dropped, examples = 0, []
    for s in smiles_list:
        try:
            smiles_to_tensor(s, atom_vocab=vocab)
        except Exception:
            dropped += 1
            if len(examples) < 5:
                examples.append(s)
    return dropped, examples

for name, data, vocab in [("QM9", qm9.data, QM9_ATOMS), ("ZINC-250k", zinc.data, ZINC_ATOMS)]:
    n = len(data)
    d, ex = count_dropped(data, vocab)
    print(f"{name:10s}: {d:>7,}/{n:>7,} dropped = {d / n:.2%}  (kept {n - d:,})")
    print(f"            e.g. {ex[:3]}")